# Take-Home Exercise 2
#### Haoxuan Lu
#### CUID: hl3921
#### Date: Apr 3, 2026

## Setup and data loading

I first load the datasets needed for the homework from the shared volume. I use PySpark so the transformations stay distributed and scalable. I also standardize some columns into numeric/date formats right away because later questions require aggregation, ranking, and date arithmetic.

The main datasets used are:

- `drivers`: driver identity information
- `races`: race-level information including race date
- `results`: final finishing positions for each driver in each race
- `pit_stops`: pit stop timing information

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

base_path = "/Volumes/gr5069/raw/f1_data/"

drivers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/drivers.csv")
)

races = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/races.csv")
    .withColumn("date", F.to_date("date"))
)

results = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/results.csv")
)

pit_stops = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/pit_stops.csv")
)

display(drivers)
display(races)
display(results)
display(pit_stops)

## How the setup code works

- `spark.read.option("header", True).option("inferSchema", True).csv(...)` loads each CSV and asks Spark to detect data types automatically.
- `F.to_date("date")` converts the race date from text into a true date column, which is important for calculating age and ordering races chronologically.
- I import `functions as F` for column operations and `Window` for ranking and cumulative calculations later in the assignment.

## Q1. Average pit stop time each driver spent in each race

My logic is:

1. Convert pit stop duration into a numeric value in seconds. If the dataset contains `milliseconds`, I use that because it is the cleanest numeric format.
2. Group pit stop records by `raceId` and `driverId` to compute the driver's average pit stop time in that race.
3. Separately, group by `raceId` only to compute the fastest and slowest single pit stop recorded in that race.
4. Join race names so the final output is easier to read.

This gives a driver-race average while also attaching race-level fastest and slowest pit stop benchmarks.

In [0]:
pit_stops_clean = (
    pit_stops
    .withColumn("pit_stop_seconds", F.col("milliseconds") / 1000.0)
)

driver_race_avg_pit = (
    pit_stops_clean
    .groupBy("raceId", "driverId")
    .agg(
        F.avg("pit_stop_seconds").alias("avg_pit_stop_seconds"),
        F.count("*").alias("num_pit_stops")
    )
)

race_pit_extremes = (
    pit_stops_clean
    .groupBy("raceId")
    .agg(
        F.min("pit_stop_seconds").alias("fastest_pit_stop_seconds"),
        F.max("pit_stop_seconds").alias("slowest_pit_stop_seconds")
    )
)

q1 = (
    driver_race_avg_pit
    .join(race_pit_extremes, on="raceId", how="left")
    .join(races.select("raceId", "year", "name", "date"), on="raceId", how="left")
    .join(drivers.select("driverId", "forename", "surname"), on="driverId", how="left")
    .select(
        "year",
        "name",
        "date",
        "driverId",
        "forename",
        "surname",
        "num_pit_stops",
        F.round("avg_pit_stop_seconds", 3).alias("avg_pit_stop_seconds"),
        F.round("fastest_pit_stop_seconds", 3).alias("fastest_pit_stop_seconds"),
        F.round("slowest_pit_stop_seconds", 3).alias("slowest_pit_stop_seconds")
    )
    .orderBy("year", "date", "driverId")
)

display(q1)

## How the code answers Q1

- `withColumn("pit_stop_seconds", F.col("milliseconds") / 1000.0)` converts milliseconds to seconds.
- `groupBy("raceId", "driverId").agg(F.avg(...))` calculates the average pit stop time for each driver in each race.
- `groupBy("raceId").agg(F.min(...), F.max(...))` finds the fastest and slowest single pit stops in each race.
- The `join(...)` commands bring race and driver names into the result.
- `F.round(...)` makes the output cleaner to read.

### Extra credit / alternative route

An alternative route would be to use the `duration` string column if `milliseconds` were missing. That would require parsing a text time format into seconds first, which is more error-prone than using milliseconds directly.

## Q2. Rank by finishing position the average time spent at the pit stop in each race

My logic is:

1. Reuse the average pit stop time per driver per race from Question 1.
2. Join that result to the race results table using `raceId` and `driverId`.
3. Use the official finishing order column to rank drivers within each race.
4. Sort the final output so each race shows drivers in finishing order along with their average pit stop time.

This lets us compare race finish with average pit stop performance.

In [0]:
results_clean = (
    results
    .withColumn("positionOrder", F.col("positionOrder").cast("int"))
)

q2 = (
    results_clean
    .join(driver_race_avg_pit, on=["raceId", "driverId"], how="left")
    .join(races.select("raceId", "year", "name", "date"), on="raceId", how="left")
    .join(drivers.select("driverId", "forename", "surname"), on="driverId", how="left")
    .select(
        "year",
        "name",
        "date",
        "positionOrder",
        "driverId",
        "forename",
        "surname",
        F.round("avg_pit_stop_seconds", 3).alias("avg_pit_stop_seconds"),
        "num_pit_stops"
    )
    .orderBy("year", "date", "positionOrder")
)

display(q2)

## How the code answers Q2

- `results_clean` ensures `positionOrder` is numeric so sorting behaves correctly.
- `join(driver_race_avg_pit, on=["raceId", "driverId"], how="left")` matches each driver’s finishing result with that driver’s average pit stop time in the same race.
- `orderBy("year", "date", "positionOrder")` puts each race in chronological order and each driver in official finishing order.

### Extra credit / alternative route

A useful alternative would be to add a within-race rank for pit stop speed itself, then compare:
- finishing position rank
- pit stop average rank

That would make it easier to see whether quicker pit stops line up with better race outcomes.

## Q3. Fill missing driver codes

My logic is:

1. Check which drivers have a missing `code`.
2. Create a fallback code using the first three letters of the driver's surname in uppercase, which matches the example `ALO` for Alonso.
3. Keep any existing code unchanged.
4. Store the completed value in a new column so the original raw column is still preserved.

This approach is simple, transparent, and follows the naming pattern shown in the prompt.

In [0]:
drivers_with_code = (
    drivers
    .withColumn(
        "imputed_code",
        F.upper(
            F.substring(
                F.regexp_replace(F.col("surname"), "[^A-Za-z]", ""),
                1,
                3
            )
        )
    )
    .withColumn(
        "code_filled",
        F.when(F.col("code").isNull() | (F.trim(F.col("code")) == ""), F.col("imputed_code"))
         .otherwise(F.col("code"))
    )
)

q3_missing_only = (
    drivers_with_code
    .filter(F.col("code").isNull() | (F.trim(F.col("code")) == ""))
    .select(
        "driverId",
        "forename",
        "surname",
        "code",
        "imputed_code",
        "code_filled"
    )
    .orderBy("driverId")
)

display(q3_missing_only)

## How the code answers Q3

- `regexp_replace(F.col("surname"), "[^A-Za-z]", "")` removes punctuation or non-letter symbols from surnames.
- `substring(..., 1, 3)` takes the first three characters.
- `upper(...)` converts the result to the standard driver-code style.
- `when(...).otherwise(...)` fills only missing codes and leaves existing codes untouched.

### Extra credit / alternative route

Another route would be to build the code from:
- surname if it has at least 3 letters
- otherwise surname + first letter(s) of forename

That would help if a surname were unusually short. I did not do that here because most Formula 1 surnames are long enough, and the assignment example suggests a surname-based code.

## Q4. Youngest and oldest driver in each race

My logic is:

1. Join race results to driver birth dates and race dates.
2. Define `Age` as full years completed on the race date.
3. Within each race, rank drivers from youngest to oldest and from oldest to youngest.
4. Keep the youngest and oldest driver(s) for each race.

I define age as the number of completed years on the day of the race, not approximate age by year only. This is more precise because it accounts for month and day.

In [0]:
drivers_birth = drivers.withColumn("dob", F.to_date("dob"))

race_driver_age = (
    results_clean
    .join(races.select("raceId", "year", "name", "date"), on="raceId", how="left")
    .join(drivers_birth.select("driverId", "forename", "surname", "dob"), on="driverId", how="left")
    .withColumn(
        "Age",
        F.floor(F.months_between(F.col("date"), F.col("dob")) / 12)
    )
)

youngest_window = Window.partitionBy("raceId").orderBy(F.col("Age").asc(), F.col("dob").desc())
oldest_window = Window.partitionBy("raceId").orderBy(F.col("Age").desc(), F.col("dob").asc())

youngest_per_race = (
    race_driver_age
    .withColumn("youngest_rank", F.dense_rank().over(youngest_window))
    .filter(F.col("youngest_rank") == 1)
    .select(
        "raceId", "year", "name", "date",
        F.lit("youngest").alias("age_group"),
        "driverId", "forename", "surname", "dob", "Age"
    )
)

oldest_per_race = (
    race_driver_age
    .withColumn("oldest_rank", F.dense_rank().over(oldest_window))
    .filter(F.col("oldest_rank") == 1)
    .select(
        "raceId", "year", "name", "date",
        F.lit("oldest").alias("age_group"),
        "driverId", "forename", "surname", "dob", "Age"
    )
)

q4 = youngest_per_race.unionByName(oldest_per_race).orderBy("year", "date", "age_group")

display(q4)

## How the code answers Q4

- `to_date("dob")` converts the driver birth date into a proper date column.
- `months_between(F.col("date"), F.col("dob")) / 12` computes age in years with month-level precision.
- `floor(...)` converts that to completed years.
- `Window.partitionBy("raceId")` tells Spark to rank drivers separately inside each race.
- `dense_rank()` identifies the youngest and oldest driver per race.

### Extra credit / alternative route

An alternative definition of age would be decimal age, such as 24.7 years old. That may be useful analytically, but for this homework I used completed whole years because it is easier to interpret and matches how age is usually reported.

## Q5. Podium counts by driver at any given race

My logic is:

1. Use race results and race dates to order each driver’s career chronologically.
2. For each result row, create indicator columns for:
   - win
   - second place
   - third place
3. Use cumulative window sums to count how many of each podium finish the driver has at that race.
4. Also create a total podiums column by summing the three cumulative podium components.

I interpret “at any given race” as including the current race result. So if a driver wins a race, that race is counted in the cumulative win total shown on that row.

In [0]:
results_with_race = (
    results_clean
    .join(races.select("raceId", "year", "name", "date"), on="raceId", how="left")
    .join(drivers.select("driverId", "forename", "surname"), on="driverId", how="left")
)

career_window = (
    Window
    .partitionBy("driverId")
    .orderBy(F.col("date"), F.col("raceId"))
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

q5 = (
    results_with_race
    .withColumn("win_flag", F.when(F.col("positionOrder") == 1, 1).otherwise(0))
    .withColumn("second_flag", F.when(F.col("positionOrder") == 2, 1).otherwise(0))
    .withColumn("third_flag", F.when(F.col("positionOrder") == 3, 1).otherwise(0))
    .withColumn("wins", F.sum("win_flag").over(career_window))
    .withColumn("second_places", F.sum("second_flag").over(career_window))
    .withColumn("third_places", F.sum("third_flag").over(career_window))
    .withColumn("podiums", F.col("wins") + F.col("second_places") + F.col("third_places"))
    .select(
        "year",
        "name",
        "date",
        "driverId",
        "forename",
        "surname",
        "positionOrder",
        "wins",
        "second_places",
        "third_places",
        "podiums"
    )
    .orderBy("year", "date", "positionOrder")
)

display(q5)

## How the code answers Q5

- `withColumn("win_flag", ...)`, `second_flag`, and `third_flag` create one-hot indicators for podium positions.
- The `career_window` partitions by `driverId` and orders each driver’s races over time.
- `F.sum("win_flag").over(career_window)` produces a cumulative count of wins through that race.
- The same pattern creates cumulative second-place and third-place totals.
- `podiums` is the total number of top-3 finishes by that race.

### Extra credit / alternative route

A different interpretation would be “podiums before the race starts.” In that case, I would change the window frame to stop at the previous row instead of the current row:

`rowsBetween(Window.unboundedPreceding, -1)`

That would show the driver’s career podium total entering the race, rather than after the race.

## Q6. My own exploration question

### Question:
Which races had the highest average number of pit stops per driver?

### Logic:
1. Count how many pit stops each driver made in each race.
2. For each race, average those pit stop counts across drivers.
3. Sort descending to find the races with the highest average pit-stop activity.

This is interesting because it highlights races where strategy, tire degradation, or race conditions may have caused unusually frequent pit stops.

In [0]:
pit_stops_per_driver_race = (
    pit_stops
    .groupBy("raceId", "driverId")
    .agg(F.count("*").alias("pit_stops_for_driver"))
)

q6 = (
    pit_stops_per_driver_race
    .groupBy("raceId")
    .agg(
        F.avg("pit_stops_for_driver").alias("avg_pit_stops_per_driver"),
        F.max("pit_stops_for_driver").alias("max_pit_stops_by_one_driver"),
        F.countDistinct("driverId").alias("drivers_with_pit_data")
    )
    .join(races.select("raceId", "year", "name", "date"), on="raceId", how="left")
    .select(
        "year",
        "name",
        "date",
        F.round("avg_pit_stops_per_driver", 2).alias("avg_pit_stops_per_driver"),
        "max_pit_stops_by_one_driver",
        "drivers_with_pit_data"
    )
    .orderBy(F.col("avg_pit_stops_per_driver").desc(), "year", "date")
)

display(q6)

## How the code answers Q6

- `groupBy("raceId", "driverId").agg(F.count("*"))` counts each driver’s number of pit stops in a race.
- Grouping again by `raceId` and taking `avg(...)` calculates the race-level average.
- `max(...)` shows the most pit stops taken by any one driver in that race.
- `orderBy(F.col("avg_pit_stops_per_driver").desc())` ranks the most pit-stop-heavy races first.

### Extra credit / alternative route

A more advanced version of this question would compare pit-stop-heavy races by season, circuit, or weather if those variables are available in other related datasets.